In [4]:
import pandas as pd
import numpy as np
import hashlib
import json
from pathlib import Path

# Load dataset
file_path = "salary_task1_cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset Loaded Successfully!")
print("Shape:", df.shape)

Dataset Loaded Successfully!
Shape: (2000, 18)


In [5]:
# Display first 5 records
display(df.head())

,age,gender,education,experience_years,role_seniority,company_size,location_tier,skills_count,certifications,worked_remote,last_promotion_years_ago,salary_bdt,recent_project_description_length,survey_date,recent_note,survey_year,survey_month,role_seniority_encoded
0,33,Male,M.Sc,6.0,Senior,Enterprise,Tier-2,1.0,2.0,1,0.0,145185,57.0,2024-11-15,Worked on microservices deployment,2024,11,2
1,29,Male,M.Sc,9.0,Junior,Enterprise,Remote,5.0,0.0,1,0.0,121262,50.0,2024-03-08,Built ML pipeline for preprocessing,2024,3,0
2,34,Male,B.Sc+Cert,NaN,Lead,Enterprise,Remote,5.0,1.0,1,5.0,184875,50.0,2024-09-01,Experience in data cleaning and ETL,2024,9,3
3,39,Male,B.Sc,16.0,Mid,Enterprise,Tier-1,5.0,0.0,1,3.0,180105,59.0,2024-01-26,Contributed to open-source NLP repo,2024,1,1
4,29,Female,B.Sc,8.0,Junior,SME,Tier-1,5.0,2.0,0,7.0,114750,NaN,2023-12-08,Built ML pipeline for preprocessing,2023,12,0


In [6]:
# Display column names
print("Column Names:")
print(df.columns.tolist())

Column Names:
['age', 'gender', 'education', 'experience_years', 'role_seniority', 'company_size', 'location_tier', 'skills_count', 'certifications', 'worked_remote', 'last_promotion_years_ago', 'salary_bdt', 'recent_project_description_length', 'survey_date', 'recent_note', 'survey_year', 'survey_month', 'role_seniority_encoded']


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 18 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   age                                2000 non-null   int64  
 1   gender                             2000 non-null   str    
 2   education                          1900 non-null   str    
 3   experience_years                   1840 non-null   float64
 4   role_seniority                     2000 non-null   str    
 5   company_size                       2000 non-null   str    
 6   location_tier                      2000 non-null   str    
 7   skills_count                       1800 non-null   float64
 8   certifications                     1920 non-null   float64
 9   worked_remote                      2000 non-null   int64  
 10  last_promotion_years_ago           1880 non-null   float64
 11  salary_bdt                         2000 non-null   int64  
 12  rec

In [8]:
missing_values = df.isnull().sum()

print("Missing Values:")
print(missing_values)

Missing Values:
age                                    0
gender                                 0
education                            100
experience_years                     160
role_seniority                         0
company_size                           0
location_tier                          0
skills_count                         200
certifications                        80
worked_remote                          0
last_promotion_years_ago             120
salary_bdt                             0
recent_project_description_length    240
survey_date                            0
recent_note                          300
survey_year                            0
survey_month                           0
role_seniority_encoded                 0
dtype: int64


In [9]:
duplicates = df.duplicated().sum()

print("Number of duplicate rows:", duplicates)

Number of duplicate rows: 0


In [10]:
print("===== DATA INTEGRITY CHECKS =====")

print("Invalid Age:",
      ((df["age"] < 18) | (df["age"] > 70)).sum())

print("Negative Experience:",
      (df["experience_years"] < 0).sum())

print("Negative Skills:",
      (df["skills_count"] < 0).sum())

print("Negative Certifications:",
      (df["certifications"] < 0).sum())

print("Invalid Worked Remote:",
      (~df["worked_remote"].isin([0, 1])).sum())

print("Invalid Salary:",
      (df["salary_bdt"] <= 0).sum())

print("Invalid Survey Month:",
      (~df["survey_month"].between(1, 12)).sum())

print("Invalid Role Encoding:",
      (~df["role_seniority_encoded"].isin([0, 1, 2, 3])).sum())

===== DATA INTEGRITY CHECKS =====
Invalid Age: 0
Negative Experience: 0
Negative Skills: 0
Negative Certifications: 0
Invalid Worked Remote: 0
Invalid Salary: 0
Invalid Survey Month: 0
Invalid Role Encoding: 0


In [11]:
df["survey_date"] = pd.to_datetime(
    df["survey_date"],
    errors="coerce"
)

print(df["survey_date"].dtype)

datetime64[us]


In [12]:
numeric_columns = [
    "experience_years",
    "skills_count",
    "certifications",
    "last_promotion_years_ago",
    "recent_project_description_length"
]

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

print("Numerical missing values handled successfully.")

Numerical missing values handled successfully.


In [13]:
categorical_columns = [
    "education",
    "recent_note"
]

for column in categorical_columns:
    df[column] = df[column].fillna("Unknown")
    df[column] = df[column].astype(str).str.strip()

print("Categorical missing values handled successfully.")

Categorical missing values handled successfully.


In [14]:
text_columns = [
    "gender",
    "role_seniority",
    "company_size",
    "location_tier"
]

for column in text_columns:
    df[column] = df[column].astype(str).str.strip()

print("Text columns cleaned successfully.")

Text columns cleaned successfully.


In [15]:
df.insert(0, "data_version", "v1.0")

print(df.head())

  data_version  age  gender  education  experience_years role_seniority  \
0         v1.0   33    Male       M.Sc               6.0         Senior   
1         v1.0   29    Male       M.Sc               9.0         Junior   
2         v1.0   34    Male  B.Sc+Cert               7.0           Lead   
3         v1.0   39    Male       B.Sc              16.0            Mid   
4         v1.0   29  Female       B.Sc               8.0         Junior   

  company_size location_tier  skills_count  certifications  worked_remote  \
0   Enterprise        Tier-2           1.0             2.0              1   
1   Enterprise        Remote           5.0             0.0              1   
2   Enterprise        Remote           5.0             1.0              1   
3   Enterprise        Tier-1           5.0             0.0              1   
4          SME        Tier-1           5.0             2.0              0   

   last_promotion_years_ago  salary_bdt  recent_project_description_length  \
0       

In [16]:
print("Total missing values:",
      df.isnull().sum().sum())

Total missing values: 0


In [17]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [18]:
output_folder = Path("task1_outputs")
output_folder.mkdir(exist_ok=True)

csv_path = output_folder / "salary_task1_v1_processed.csv"

df.to_csv(csv_path, index=False)

print("Dataset saved successfully!")
print(csv_path)

Dataset saved successfully!
task1_outputs\salary_task1_v1_processed.csv


In [19]:
!pip install pyarrow

In [20]:
parquet_path = output_folder / "salary_task1_v1_processed.parquet"

df.to_parquet(
    parquet_path,
    index=False,
    engine="pyarrow"
)

print("Parquet file saved successfully!")

Parquet file saved successfully!


In [21]:
feather_path = output_folder / "salary_task1_v1_processed.feather"

df.to_feather(
    feather_path
)

print("Feather file saved successfully!")

Feather file saved successfully!


In [22]:
integrity_report = {
    "original_rows": 2000,
    "processed_rows": len(df),
    "original_columns": 18,
    "processed_columns": len(df.columns),
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_values": int(df.isnull().sum().sum()),
    "invalid_age": int(((df["age"] < 18) | (df["age"] > 70)).sum()),
    "invalid_salary": int((df["salary_bdt"] <= 0).sum()),
    "invalid_survey_month": int(
        (~df["survey_month"].between(1, 12)).sum()
    )
}

print("===== FINAL INTEGRITY REPORT =====")

for key, value in integrity_report.items():
    print(f"{key}: {value}")

===== FINAL INTEGRITY REPORT =====
original_rows: 2000
processed_rows: 2000
original_columns: 18
processed_columns: 19
duplicate_rows: 0
missing_values: 0
invalid_age: 0
invalid_salary: 0
invalid_survey_month: 0


In [23]:
import hashlib
import json

file_hash = hashlib.sha256(
    csv_path.read_bytes()
).hexdigest()

manifest = {
    "project": "SalaryInsight: Predictive Modeling for Informed Salary Predictions",
    "task": "Task 1",
    "version": "v1.0",
    "source_file": "salary_task1_cleaned.csv",
    "processed_file": "salary_task1_v1_processed.csv",
    "rows": len(df),
    "columns": len(df.columns),
    "missing_values_after_processing": int(df.isnull().sum().sum()),
    "duplicate_rows_after_processing": int(df.duplicated().sum()),
    "sha256": file_hash,
    "transformations": [
        "Converted survey_date to datetime",
        "Filled numerical missing values using median",
        "Filled categorical missing values using Unknown",
        "Trimmed text values",
        "Added data_version column"
    ]
}

manifest_path = output_folder / "task1_manifest_v1.json"

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("Version manifest created successfully!")

Version manifest created successfully!
